# Step 3 -- Ancestral GOEA with EAGLE (Extant and Ancestral Gene List Enrichment)

In [21]:
import os
import subprocess

Path definitions:

In [22]:
mini_dataset_path = '/work/FAC/FBM/DBC/cdessim2/default/summer_school2026/mini_dataset'

nwk_fn = os.path.join(mini_dataset_path, 'species_tree.nwk')  # might need to be the species_tree_checked.nwk from FastOMA output
oxml_fn = os.path.join(mini_dataset_path, 'fastoma_output', 'FastOMA_HOGs.orthoxml')

obo_fn = '../step1_annotation/go-basic.obo'
hogprop_results_fn = '../step2_propagation/hogprop_output.h5'
#'../step2_propagation/hogprop_output.h5'

In [23]:
hogprop_results_fn

'../step2_propagation/hogprop_output.h5'

In [24]:
!cat {nwk_fn}

((Crocodylus_porosus:1,(((((((Aptenodytes_forsteri :1, Oreotrochilus_melanogaster :1)Neoaves_4c:1, Grus_americana :1)Neoaves_3c:1, Taeniopygia_guttata :1)Neoaves_2a:1, Columba_livia :1)Neoaves_1a:1, Phoenicopterus_ruber :1)Neoaves:1, Gallus_gallus :1)Neognathae:1, Struthio_camelus:1)Aves:0)Archosauria:1, Homo_sapiens:1);


We need the newick string from FastOMA results (species_tree_checked.nwk) as this has the same internal node names as the OrthoXML. Below we create a variable called nwk where we copy and paste the tree string.

In [25]:
nwk = '((Crocodylus_porosus:1,(((((((Aptenodytes_forsteri :1, Oreotrochilus_melanogaster :1)Neoaves_4c:1, Grus_americana :1)Neoaves_3c:1, Taeniopygia_guttata :1)Neoaves_2a:1, Columba_livia :1)Neoaves_1a:1, Phoenicopterus_ruber :1)Neoaves:1, Gallus_gallus :1)Neognathae:1, Struthio_camelus:1)Aves:0)Archosauria:1, Homo_sapiens:1)internal_0;'

In [26]:
#write species tree newick to current path

with open('species_tree.nwk', 'wt') as fp:
    print(nwk, file=fp)

# Method to run enrichment on single branch

## Running GO enrichment on a single branch

We will begin by performing GO enrichment analysis on a single branch of the bird phylogeny.

In this example, we compare the ancestral node **Archoasauria** (ancestral to all birds and crocodiles) to **Aves**, a large clade containing the modern bird species. By examining the HOGs inferred to be gained, duplicated, lost, or retained along this branch, we can identify biological functions that may have been important during the diversification of Neoaves.

EAGLE performs enrichment analysis using:

- the species tree,
- the HOG structure inferred by FastOMA,
- functional annotations propagated with HogProp,
- and the Gene Ontology hierarchy.

The `--parent_name` and `--child_name` parameters define the branch of interest. EAGLE will automatically determine which evolutionary events occurred between these two nodes and test whether particular GO terms are overrepresented among the affected HOGs.

For larger analyses, EAGLE can also be run on all branches of a phylogeny, but starting with a single branch makes it easier to understand the workflow and inspect the results.

In [27]:
%%bash
which python
which jupyter-lab
which eagle
which hogprop

/users/nglover/.conda/envs/sib_course/bin/python
/users/nglover/.conda/envs/sib_course/bin/jupyter-lab
/users/nglover/.conda/envs/sib_course/bin/eagle
/users/nglover/.conda/envs/sib_course/bin/hogprop


In [28]:
import os

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [53]:
os.makedirs("eagle_branch_path_results", exist_ok=True)

OBO = obo_fn
ORTHOXML = oxml_fn
TREE = "species_tree.nwk"
HOGPROP = hogprop_results_fn
OUTDIR = "eagle_branch_path_results"

cmd = [
    "eagle",
    "--obo", OBO,
    "--oxml", ORTHOXML,
    "--nwk", TREE,
    "--hogprop_results", HOGPROP,
    "--results", OUTDIR,
    "--include_genelist",
    "--skip_terminal",
    "--write_extant_genelist",
    "--parent_name", "Archosauria",
    "--child_name", "Aves"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)

Loaded data in 16.89493 seconds
DONE! Took 622.142356s.



In [52]:
%%bash
sed -n '118,132p' \
/work/FAC/FBM/DBC/cdessim2/default/summer_school2026/mini_dataset/function_test/gits/ancestral_goea/eagle/goea.py | cat -te

                        # level = r["annot_genome"].encode("ascii")$
                        # x = list($
                        #     self._annot_db.root.HogLevel.where($
                        #         "(ID == {}) & (Level == {})".format(hog_id, level)$
                        #     )$
                        # )$
^I                    hog_id = r["gene_id"].encode("ascii")$
                        level = r["annot_genome"].encode("ascii")$
$
                        x = list($
                            self._annot_db.root.HogLevel.where($
                            "(ID == hog_id) & (Level == level)",$
                            condvars={$
                               "hog_id": hog_id,$
                               "level": level,$


In [36]:
import tables
import pandas as pd

#natasha file
h5 = tables.open_file("../step2_propagation/hogprop_output.h5") 

df = pd.DataFrame.from_records(
    h5.root.HogLevel[:10]
)

h5.close()

df

,Fam,ID,Level
0,1,b'HOG:0000001',b'internal_0'
1,1,b'HOG:0000001',b'Archosauria'
2,1,b'HOG:0000001',b'Aves'
3,1,b'HOG:0000001',b'Neognathae'
4,1,b'HOG:0000001',b'Neoaves'
5,1,b'HOG:0000001',b'Neoaves_3c'
6,1,b'HOG:0000001',b'Neoaves_4c'
7,2,b'HOG:0000002',b'internal_0'
8,2,b'HOG:0000002',b'Archosauria'
9,2,b'HOG:0000002',b'Aves'


In [47]:
#alex file

h5 = tables.open_file("../step2_propagation/hogprop_output.h5")

hog_id = b"HOG:0000001"
level = b"Aves"

list(
    h5.root.HogLevel.where(
        "(ID == hog_id) & (Level == level)",
        condvars={
            "hog_id": hog_id,
            "level": level
        }
    )
)

[/HogLevel.row (Row), pointing to row #117632]

In [41]:
 import tables

h5 = tables.open_file(
    "/work/FAC/FBM/DBC/cdessim2/default/awarwick/sib_course/step2_propagation/hogprop_output.h5"
)

list(h5.root.HogLevel.where("Fam == 1"))[:3]

[/HogLevel.row (Row), pointing to row #6,
 /HogLevel.row (Row), pointing to row #6,
 /HogLevel.row (Row), pointing to row #6]

In [46]:
import tables
import numexpr
import numpy as np

print("tables:", tables.__version__)
print("numexpr:", numexpr.__version__)
print("numpy:", np.__version__)

tables: 3.9.1
numexpr: 2.8.7
numpy: 1.26.1


In [16]:
import tables

h5 = tables.open_file("../step2_propagation/hogprop_results/output.h5")

print(h5.root.HogLevel)
print(h5.root.HogLevel.colnames)
print(h5.root.HogLevel[0])
print(h5.root.HogLevel.nrows)
h5.close()


/HogLevel (Table(117633,)fletcher32, shuffle, zlib(6)) 'similar to /HogLevel in pyomadb'
['Fam', 'ID', 'Level']
(1, b'HOG:0000001', b'internal_0')
117633


In [18]:
import tables

src = tables.open_file("../step2_propagation/hogprop_results/output.h5", mode="r")
dst = tables.open_file("../step2_propagation/hogprop_output.h5", mode="a")

# copy table
src.root.HogLevel.copy(dst.root, newname="HogLevel")

src.close()
dst.close()

In [20]:
import tables

h5 = tables.open_file("../step2_propagation/hogprop_output.h5")

for node in h5.walk_nodes("/"):
    print(node)

h5.close()

/ (RootGroup) ''
/Annotations (Group) ''
/HOGAnnotations (Group) ''
/HogLevel (Table(117633,)fletcher32, shuffle, zlib(6)) 'similar to /HogLevel in pyomadb'
/Annotations/GeneOntology_HOGPROP (Table(17566826,)fletcher32, shuffle, zlib(6)) 'HOGPROP gene ontology extant predictions'
/HOGAnnotations/GeneOntology (Table(11799967,)fletcher32, shuffle, zlib(6)) 'HOGPROP gene ontology ancestral predictions'


In [11]:
import h5py

f = h5py.File("../step2_propagation/hogprop_results/output.h5", "r")
print(list(f.keys()))
f.close()

['HogLevel', '_i_HogLevel']


In [12]:
import tables

h5 = tables.open_file("/work/FAC/FBM/DBC/cdessim2/default/awarwick/sib_course/step2_propagation/hogprop_output.h5")
print(h5)
h5.close()

/work/FAC/FBM/DBC/cdessim2/default/awarwick/sib_course/step2_propagation/hogprop_output.h5 (File) ''
Last modif.: '2026-05-21T13:48:02+00:00'
Object Tree: 
/ (RootGroup) ''
/HogLevel (Table(117633,)fletcher32, shuffle, zlib(6)) 'similar to /HogLevel in pyomadb'
/Annotations (Group) ''
/Annotations/GeneOntology_HOGPROP (Table(16374573,)fletcher32, shuffle, zlib(6)) 'HOGPROP gene ontology extant predictions'
/HOGAnnotations (Group) ''
/HOGAnnotations/GeneOntology (Table(11116006,)fletcher32, shuffle, zlib(6)) 'HOGPROP gene ontology ancestral predictions'



---

Alternatively, submit the script `sbatch eagle.sh` in order to run on every evolutionary event set of genes in the tree (internal branches only).

---

Now, to read the results we can analyse individual files / branches, e.g., 

In [6]:
from eagle.results import ResultsReader

/users/nglover/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
!ls eagle_branch_path_results

Archosauria_Aves_duplicated.extant_genelist.tsv.gz
Archosauria_Aves_duplicated.tsv.gz
Archosauria_Aves_gained.extant_genelist.tsv.gz
Archosauria_Aves_gained.tsv.gz
Archosauria_Aves_lost.extant_genelist.tsv.gz
Archosauria_Aves_lost.tsv.gz
Archosauria_Aves_retained.extant_genelist.tsv.gz
Archosauria_Aves_retained.tsv.gz
Aves_Neoaves_duplicated.extant_genelist.tsv.gz
Aves_Neoaves_duplicated.tsv.gz
Aves_Neoaves_gained.extant_genelist.tsv.gz
Aves_Neoaves_gained.tsv.gz
Aves_Neoaves_lost.extant_genelist.tsv.gz
Aves_Neoaves_lost.tsv.gz
Aves_Neoaves_retained.extant_genelist.tsv.gz
Aves_Neoaves_retained.tsv.gz


In [8]:
r = ResultsReader('eagle_branch_path_results/Archosauria_Aves_duplicated.tsv.gz')

In [9]:
r.header

{'eagle_version': '0.0.2',
 'time_run': '2026-06-09 10:37:43.105678',
 'time_taken': '98.349827',
 'branch_type': 'None',
 'event_type': 'duplicated',
 'parent_name': 'Archosauria',
 'child_name': 'Aves'}

In [10]:
df = r.read()

In [11]:
df.head()

,GO_ID,GO_name,GO_aspect,GO_depth,p_uncorrected,p_bonferroni,p_fdr_bh,study_count,study_size,study_ratio,study_proportion,population_count,population_size,population_ratio,population_proportion,fold_change,study_entries_with_go_term
0,GO:0086046,membrane depolarization during SA node cell ac...,BP,7,0.000008,0.049574,0.004507,4,194,4 / 194,0.020619,10,13456,10 / 13456,0.000743,27.744330,HOG:0007817.1a_2@Archosauria;HOG:0007819_2@Arc...
1,GO:0043194,axon initial segment,CC,2,0.000008,0.048656,0.004507,5,188,5 / 188,0.026596,21,13624,21 / 13624,0.001541,17.254306,HOG:0007817.1a_2@Archosauria;HOG:0007819_2@Arc...
2,GO:0045956,positive regulation of calcium ion-dependent e...,BP,8,0.000002,0.014501,0.004507,6,194,6 / 194,0.030928,28,13456,28 / 13456,0.002081,14.863034,HOG:0005181.1a_2@Archosauria;HOG:0007817.1a_2@...
3,GO:1903307,positive regulation of regulated secretory pat...,BP,7,0.000005,0.031290,0.004507,7,194,7 / 194,0.036082,48,13456,48 / 13456,0.003567,10.115120,HOG:0005181.1a_2@Archosauria;HOG:0007817.1a_2@...
4,GO:0045921,positive regulation of exocytosis,BP,6,0.000004,0.027074,0.004507,10,194,10 / 194,0.051546,111,13456,111 / 13456,0.008249,6.248723,HOG:0000839.1d_2@Archosauria;HOG:0005181.1a_2@...


Or we can read many at once:

In [20]:
from eagle.results import parse_to_concat
import gzip

In [56]:
goea_path = 'eagle_results'
fns = list(map(lambda x: os.path.join(goea_path, x),
               filter(lambda x: not x.endswith('.extant_genelist.tsv.gz'),
                      os.listdir(goea_path))))
df = parse_to_concat(*fns)

with gzip.open('eagle_results_with_ancestral_genes.tsv.gz', 'wt') as fp:
    df.to_csv(fp, sep='\t', index=False)

with gzip.open('eagle_results.tsv.gz', 'wt') as fp:
    header = [x for x in list(df) if x != 'study_entries_with_go_term']
    df[header].to_csv(fp, sep='\t', index=False)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 32/32 [00:12<00:00,  2.60it/s]


In the previous step, we identified GO terms that are significantly enriched among HOGs that were gained, duplicated, retained, or lost along the branch leading from Aves to Neoaves. However, enrichment analyses often produce long lists of highly redundant GO terms because many terms describe related biological processes.

To simplify interpretation, we will first reduce GO term redundancy using GO-Figure! and then explore the major functional themes associated with each event type.

## Step 1. Filter significant enrichments

For each event type, retain only significantly enriched GO terms:

In [57]:
sig_df = df[df['p_bonferroni']<=0.05]

How many significantly enriched GO terms are associated with gained, duplicated, retained, and lost HOGs?

## Step 2. Reduce redundancy with GO-Figure!

Use GO-Figure! to cluster related GO terms and generate summary plots.

For each event type:

- gained
- duplicated
- retained
- lost

generate:

A GO-Figure! input file
A GO-Figure! summary plot

In [23]:
sig_df.columns

Index(['GO_ID', 'GO_name', 'GO_aspect', 'GO_depth', 'p_uncorrected',
       'p_bonferroni', 'p_fdr_bh', 'study_count', 'study_size', 'study_ratio',
       'study_proportion', 'population_count', 'population_size',
       'population_ratio', 'population_proportion', 'fold_change',
       'study_entries_with_go_term', 'branch_type', 'event_type',
       'parent_name', 'child_name'],
      dtype='object')

In [28]:
import os

os.makedirs("gofigure_inputs", exist_ok=True)
os.makedirs("gofigure_outputs", exist_ok=True)

#needed for go-figure
- python -m pip install scikit-learn
- python -m pip install seaborn

In [48]:
event_types = ['gained', 'lost', 'retained', 'duplicated']

for event_type in event_types:
    tmp_df = sig_df[
        (sig_df["event_type"] == event_type) &
        (sig_df['parent_name']=='Archosauria') &
        (sig_df['child_name']=='Aves')
    ]
    
    gofigure_input = tmp_df[["GO_ID", "p_bonferroni"]]
    
    gofigure_input.to_csv(
        "./gofigure_inputs/{}.tsv".format(event_type),
        sep="\t",
        index=False,
        header=False
    )

In [50]:
import subprocess

for event_type in event_types:

    cmd = [
        "python",
        "../gits/GO-Figure/gofigure.py",
        "-i", "gofigure_inputs/{}.tsv".format(event_type),
        "-o", "gofigure_outputs/gofigure_{}.tsv".format(event_type)
    ]
    print(cmd)
    subprocess.run(cmd, check=True)

['python', '../gits/GO-Figure/gofigure.py', '-i', 'gofigure_inputs/gained.tsv', '-o', 'gofigure_outputs/gofigure_gained.tsv']


Starting calculations

Calculating biological process
Output for biological processfound in: gofigure_outputs/gofigure_gained.tsv/biological_process_gofigure_outputs_gofigure_gained.tsv.png

Calculating molecular function
Output for molecular functionfound in: gofigure_outputs/gofigure_gained.tsv/molecular_function_gofigure_outputs_gofigure_gained.tsv.png

Calculating cellular component
No GO terms for ontology found: cellular_component
Finished!


['python', '../gits/GO-Figure/gofigure.py', '-i', 'gofigure_inputs/lost.tsv', '-o', 'gofigure_outputs/gofigure_lost.tsv']


Starting calculations

Calculating biological process
Output for biological processfound in: gofigure_outputs/gofigure_lost.tsv/biological_process_gofigure_outputs_gofigure_lost.tsv.png

Calculating molecular function
Output for molecular functionfound in: gofigure_outputs/gofigure_lost.tsv/molecular_function_gofigure_outputs_gofigure_lost.tsv.png

Calculating cellular component
Output for cellular componentfound in: gofigure_outputs/gofigure_lost.tsv/cellular_component_gofigure_outputs_gofigure_lost.tsv.png
Finished!


['python', '../gits/GO-Figure/gofigure.py', '-i', 'gofigure_inputs/retained.tsv', '-o', 'gofigure_outputs/gofigure_retained.tsv']


Starting calculations

Calculating biological process
Output for biological processfound in: gofigure_outputs/gofigure_retained.tsv/biological_process_gofigure_outputs_gofigure_retained.tsv.png

Calculating molecular function
No GO terms for ontology found: molecular_function

Calculating cellular component
/users/nglover/.conda/envs/sib_course/lib/python3.9/site-packages/sklearn/manifold/_mds.py:166: RuntimeWarning: invalid value encountered in scalar divide
  old_stress = stress / dis
/users/nglover/.conda/envs/sib_course/lib/python3.9/site-packages/sklearn/manifold/_mds.py:162: RuntimeWarning: invalid value encountered in scalar divide
  if (old_stress - stress / dis) < eps:
/users/nglover/.conda/envs/sib_course/lib/python3.9/site-packages/sklearn/manifold/_mds.py:166: RuntimeWarning: invalid value encountered in scalar divide
  old_stress = stress / dis
/users/nglover/.conda/envs/sib_course/lib/python3.9/site-packages/sklearn/manifold/_mds.py:162: RuntimeWarning: invalid value enco

['python', '../gits/GO-Figure/gofigure.py', '-i', 'gofigure_inputs/duplicated.tsv', '-o', 'gofigure_outputs/gofigure_duplicated.tsv']


Starting calculations

Calculating biological process
Output for biological processfound in: gofigure_outputs/gofigure_duplicated.tsv/biological_process_gofigure_outputs_gofigure_duplicated.tsv.png

Calculating molecular function
No GO terms for ontology found: molecular_function

Calculating cellular component
Output for cellular componentfound in: gofigure_outputs/gofigure_duplicated.tsv/cellular_component_gofigure_outputs_gofigure_duplicated.tsv.png
Finished!


## Step 3. Examine major biological themes


Inspect the GO-Figure! plots and identify the most prominent functional categories.

Questions

- Which biological processes or cellular components appear most strongly enriched among gained HOGs?
- Which themes are associated with duplicated HOGs?
- Which functions are predominantly lost?
- Are there any biological themes that seem particularly relevant to bird evolution?

In [62]:

sig_df[sig_df['GO_name'].str.contains('intermediate filament cytoskeleton')]

,GO_ID,GO_name,GO_aspect,GO_depth,p_uncorrected,p_bonferroni,p_fdr_bh,study_count,study_size,study_ratio,...,population_count,population_size,population_ratio,population_proportion,fold_change,study_entries_with_go_term,branch_type,event_type,parent_name,child_name
656,GO:0045111,intermediate filament cytoskeleton,CC,6,3.076302e-69,2.098345e-65,1.140405e-67,106,473,106 / 473,...,365,13475,365 / 13475,0.027087,8.273335,HOG:0002308.1au.14b.6b.2b_6@Neoaves_1a;HOG:000...,internal,lost,Neoaves_1a,Neoaves_2a
1076,GO:0045111,intermediate filament cytoskeleton,CC,6,1.202346e-61,8.376747e-58,3.932745e-60,110,412,110 / 412,...,542,13660,542 / 13660,0.039678,6.728944,HOG:0002221.1d_2&23356158053824@Aves;HOG:00023...,internal,lost,Aves,Neognathae
3046,GO:0045111,intermediate filament cytoskeleton,CC,6,6.723169e-43,4.376111e-39,5.402606e-41,102,631,102 / 631,...,458,13781,458 / 13781,0.033234,4.863916,HOG:0002308.1au.14b_4&23210491411232@Neognatha...,internal,lost,Neognathae,Neoaves
5270,GO:0045111,intermediate filament cytoskeleton,CC,6,2.902338e-30,1.929764e-26,2.717978e-28,59,313,59 / 313,...,413,13495,413 / 13495,0.030604,6.159288,HOG:0002308.1au.14b.6b_5&23114211293936@Neoave...,internal,lost,Neoaves,Neoaves_1a
8835,GO:0045111,intermediate filament cytoskeleton,CC,6,2.048834e-20,1.705039e-16,3.755593e-19,103,878,103 / 878,...,604,13624,604 / 13624,0.044334,2.646125,HOG:0000787_1&22946410080720@Archosauria;HOG:0...,internal,lost,Archosauria,Aves
11448,GO:0045111,intermediate filament cytoskeleton,CC,6,1.069788e-15,7.337676e-12,8.272465e-15,71,1288,71 / 1288,...,274,13667,274 / 13667,0.020048,2.749572,HOG:0000785_1&22748233146128@Neoaves_2a;HOG:00...,internal,lost,Neoaves_2a,Neoaves_3c
16781,GO:0045111,intermediate filament cytoskeleton,CC,6,6.367088e-11,4.902021e-07,1.795360e-09,36,651,36 / 651,...,203,12580,203 / 12580,0.016137,3.426937,HOG:0001045_1&22947374139376@Neoaves_3c;HOG:00...,internal,lost,Neoaves_3c,Neoaves_4c
20984,GO:0045111,intermediate filament cytoskeleton,CC,6,3.973676e-08,4.000299e-04,7.859134e-07,23,843,23 / 843,...,82,10788,82 / 10788,0.007601,3.589445,HOG:0000787_1@internal_0;HOG:0001045_1@interna...,internal,duplicated,internal_0,Archosauria
